In [1]:
pip install ollama sentence-transformers faiss-cpu pypdf2

Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
import faiss
import numpy as np
import pickle
import ollama
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer


In [12]:
# 1. SETTINGS
PDF_FILE = "Attention-is-all-you-need-Paper.pdf"  # Put your PDF filename here
DB_PATH = "local_vectors.index"
MODEL_NAME = "llama3.2:1b"
EMBED_MODEL = SentenceTransformer('all-MiniLM-L6-v2') # Runs locally on CPU/GPU

In [14]:
def ingest_pdf(file_path):
    print(f"📖 Extracting text from {file_path}...")
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()

    # CHUNKING (Recursive-style logic: split by 500 chars)
    chunks = [text[i:i+500] for i in range(0, len(text), 400)]
    print(f"✂️ Created {len(chunks)} chunks.")

    # EMBEDDING (Local)
    print("🔢 Generating local embeddings...")
    embeddings = EMBED_MODEL.encode(chunks)
    
    # VECTOR DB (FAISS)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings).astype('float32'))

    # Save data locally
    faiss.write_index(index, DB_PATH)
    with open("chunks.pkl", "wb") as f:
        pickle.dump(chunks, f)
    print("✅ Local Database Ready!")

In [15]:
def ask_local_rag(question):
    # Load DB
    index = faiss.read_index(DB_PATH)
    with open("chunks.pkl", "rb") as f:
        chunks = pickle.load(f)

    # RETRIEVAL
    question_embedding = EMBED_MODEL.encode([question])
    distances, indices = index.search(np.array(question_embedding).astype('float32'), k=3)
    
    context = "\n\n".join([chunks[i] for i in indices[0]])

    # GENERATION (Using Local Ollama)
    print("🤖 Local LLM is thinking...")
    response = ollama.chat(model=MODEL_NAME, messages=[
        {'role': 'system', 'content': 'Answer based ONLY on this context: ' + context},
        {'role': 'user', 'content': question},
    ])
    
    return response['message']['content']

if __name__ == "__main__":
    # Check if we need to ingest the PDF first
    if not os.path.exists(DB_PATH):
        ingest_pdf(PDF_FILE)
    
    while True:
        query = input("\n❓ Ask your PDF (or 'q' to quit): ")
        if query.lower() == 'q': break
        answer = ask_local_rag(query)
        print(f"\nAI Answer:\n{answer}")

📖 Extracting text from Attention-is-all-you-need-Paper.pdf...
✂️ Created 82 chunks.
🔢 Generating local embeddings...
✅ Local Database Ready!
🤖 Local LLM is thinking...

AI Answer:
In the context of artificial intelligence and deep learning, "attention" refers to a mechanism used in neural networks to selectively focus on certain parts of the input data when making predictions or decisions.

In traditional neural networks, each unit (neuron) receives inputs from all other units and combines them using simple rules to produce an output. This is called "feedforward" processing.

However, this can lead to several issues:

1. **Insufficient information**: If a unit only considers the input from one or two neighboring units, it may miss important context or relationships between different parts of the data.
2. **Overemphasis on noise**: Units that are too focused on their own internal state (e.g., forgetting about previous inputs) can introduce significant errors in the output.

To address t